## Model 0: baseline dummy classifier

First, let us download the data set from Kaggle (if we haven't already done so):

In [1]:
import os
import shutil

import kagglehub
import pandas as pd

if not os.path.isdir("nlp-getting-started"):
    path = kagglehub.competition_download("nlp-getting-started")
    shutil.move(path, "nlp-getting-started")

df_train = pd.read_csv("nlp-getting-started/train.csv")
df_test = pd.read_csv("nlp-getting-started/test.csv")

X_train = df_train.drop(columns=["target"])
y_train = df_train["target"]
X_test = df_test # No corresponding target column

Now we can see the data:

In [2]:
df_train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
df_test

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...


Now that that's done, let's create a dummy classifier with the default strategy for this data:

In [4]:
from sklearn.dummy import DummyClassifier

classifier = DummyClassifier()
classifier.fit(X_train, y_train)
y_test = classifier.predict(df_test)

df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
df_result.set_index("id", inplace=True)
df_result.to_csv("0_dummy.csv")

Uploading this to Kaggle gives us a score of 57.033% - approximately a coin-flip. This would indicate that the test data is fairly balanced. And if we analyze the training data, we find that it is as well.

In [5]:
y_train.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

With that in mind, there will be no need to use F1 scores or other complicated metrics when performing cross validation or grid searches later on.

## Model 1: Simple bag-of-words with Random Forest

Now, let's train an actual model on this data. First, let's create a bag-of-words vectorizer for the tweets. By default, the Scikit-Learn tokenizer for `CountVectorizer` is rather simplistic, and performs no stemming nor is it aware of things like URLs. To that end, we'll create a custom one based on the NLTK `SnowballStemmer` and `TweetTokenizer`:

In [6]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords") # ICYNI

STEMMER = SnowballStemmer(language = "english")
TOKENIZER = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
URL_RE = re.compile(r"^(https?://|www\.)")
HAS_AZ_RE = re.compile(r"[a-z]")

def stem_tokenize(text):
    tokens = TOKENIZER.tokenize(text)
    stems = []
    
    for tok in tokens:
        #remove urls
        if URL_RE.match(tok):
            continue

        # This removes the #, I am not sure if we want to remove so the words in # get tokenized like normal,
        # or if we want to some way signify that there is a # in there
        tok = tok.lstrip("#")

        # must have char a-z
        if not HAS_AZ_RE.search(tok):
            continue

        stems.append(STEMMER.stem(tok))

    return stems

base_stop_words = set(stopwords.words("english"))

keep_words = {
    'not', 'no', 'nor',
    'up', 'down', 'out', 'off',
    'after', 'before', 'during',
    'again', 'more', 'all',
    'over', 'only'
}
base_stop_words -= keep_words

STOP_WORDS = sorted({stop_w for words in base_stop_words for stop_w in stem_tokenize(words)})

bow_vectorizer = CountVectorizer(
    tokenizer=stem_tokenize,
    token_pattern=None,      
    lowercase=False,    
    stop_words=STOP_WORDS
)

/home/nicholas/.local/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/nicholas/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/nicholas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Now that we have our vectorizer, let's train and cross-validate a simple Random Forest model with the default hyperparameters on it. For now, we'll ignore the keyword and location columns.

In [9]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

ct = make_column_transformer(
    (bow_vectorizer, "text"),
    ("drop", ["id", "keyword", "location"])
)
classifier = make_pipeline(ct, RandomForestClassifier(n_jobs=-1, random_state=123))

pd.DataFrame(cross_validate(classifier, X_train, y_train, cv=5, return_train_score=True)).T

,0,1,2,3,4
fit_time,3.807651,3.944567,4.164336,3.929476,4.110416
score_time,0.347206,0.384647,0.371332,0.365865,0.503453
test_score,0.718976,0.582403,0.604071,0.620894,0.710250
train_score,0.986207,0.985878,0.988506,0.987194,0.985388


As we can see from the cross-validation results, we're overfitting _a lot_, but that's fine, right now, we only want a model better than the dummy classifier. Let's go ahead and send it to Kaggle.

In [12]:
classifier.fit(X_train, y_train)
y_test = classifier.predict(df_test)

df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
df_result.set_index("id", inplace=True)
df_result.to_csv("1_randomforest.csv")

79.282% - Much better than the dummy model, but we could probably